# Cross-Dataset Generalisation in Fake News Detection
## One-click Colab runner

**Runtime:** GPU recommended (Runtime → Change runtime type → T4 GPU)  
**Estimated time:**
- LogReg full pipeline: ~5 min  
- DistilBERT all 3 datasets × 3 seeds: ~3–4 hours on T4  

Run cells top-to-bottom. Each section is independently re-runnable.

## 0. Setup

In [ ]:
# Clone the repo (replace with your actual repo URL after pushing)
import os

REPO_URL = ""  # e.g. https://github.com/yourname/fake-news-generalization.git
REPO_DIR = "/content/fake-news-generalization"

if REPO_URL:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    # Upload the repo zip manually to /content/ and unzip, or mount Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # %cd /content/drive/MyDrive/fake-news-generalization
    print("Set REPO_URL or mount Drive manually.")

In [ ]:
!pip install -r requirements.txt -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Verify dataset access

In [ ]:
from src.data import get_dataset, DATASETS

for ds in DATASETS:
    try:
        df = get_dataset(ds, 'train')
        print(f"✓ {ds}: n={len(df)}  fake={df.label.mean():.2%}")
    except Exception as e:
        print(f"✗ {ds}: FAILED — {e}")

## 2. Phase 1A — Train LogReg baselines

In [ ]:
SEEDS = [42, 43, 44]  # reduce to [42] to save time

from src.train import train_logreg

for seed in SEEDS:
    for ds in DATASETS:
        print(f"\n--- LogReg  dataset={ds}  seed={seed} ---")
        metrics = train_logreg(ds, seed=seed)
        print(f"val f1={metrics['val_macro_f1']}  acc={metrics['val_accuracy']}")

## 2. Phase 1B — Train DistilBERT baselines (GPU required)

In [ ]:
from src.train import train_distilbert

for seed in SEEDS:
    for ds in DATASETS:
        print(f"\n--- DistilBERT  dataset={ds}  seed={seed} ---")
        metrics = train_distilbert(ds, seed=seed)
        print(metrics)

## 3. Phase 1C — Build 3×3 evaluation matrices

In [ ]:
from src.evaluate import build_matrix

logreg_matrix    = build_matrix('logreg',    seeds=SEEDS)
distilbert_matrix = build_matrix('distilbert', seeds=SEEDS)

In [ ]:
# Display heatmaps inline
from IPython.display import Image, display
display(Image('results/figures/logreg_f1_heatmap.png'))
display(Image('results/figures/distilbert_f1_heatmap.png'))

## 4. Phase 2 — Analysis

Check in with supervisor/advisor before running Phase 3.

In [ ]:
from src.analysis import dataset_statistics, source_leakage_probe, logreg_top_features, error_analysis

dataset_statistics()
source_leakage_probe()

In [ ]:
# Feature inspection for each dataset
for ds in DATASETS:
    logreg_top_features(ds, n=20)

In [ ]:
# Error analysis on the worst off-diagonal pairs
for train_ds in DATASETS:
    for test_ds in DATASETS:
        if train_ds != test_ds:
            error_analysis(train_ds, test_ds, model_type='logreg')

## 5. Phase 3 — Interventions

In [ ]:
# Intervention 1: Multi-source training
from src.interventions import multi_source_logreg, multi_source_distilbert

multi_results = {}
for held_out in DATASETS:
    for seed in SEEDS:
        r = multi_source_logreg(held_out, seed=seed)
        multi_results[f'logreg_held{held_out}_s{seed}'] = r
        print(f"multi_source/logreg held={held_out} seed={seed}: f1={r['macro_f1']}")

In [ ]:
# Intervention 2: Entity masking
from src.interventions import masking_logreg

for train_ds in DATASETS:
    for test_ds in DATASETS:
        if train_ds != test_ds:
            r = masking_logreg(train_ds, test_ds)
            print(f"masking/logreg {train_ds}→{test_ds}: f1={r['macro_f1']}")

## 6. Save results to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results/ /content/drive/MyDrive/fake-news-results/
print("Uncomment and run to save results to Drive.")